Imports pandas for data handling and urllib.request to fetch remote dataset files.

In [1]:
import pandas as pd
from urllib import request

Download the playlist dataset from the remote URL and load it into memory for parsing. Well... Why not?!

In [3]:
# get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

Parse the downloaded dataset into lines and skip the first two metadata lines.

In [4]:
# parse the playlist data file. Skip the first two lines as they only contain metadata
lines = data.read().decode('utf-8').split('\n')[2:]

Filter out playlists that contain only a single song (not useful for training sequence models).

In [5]:
# remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

Load the song metadata (ids, titles, artists) into a pandas DataFrame and index by id for lookup.

In [6]:
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode('utf-8').split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns=['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

Quickly inspect the first two playlists to ensure the parsing looks correct.

In [7]:
# let's inspect the playlists list
print( 'Playlist #1\n ', playlists[0], '\n')
print( 'Playlist #2\n ', playlists[1])

Playlist #1
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117', '

Import gensim's Word2Vec — this prepares the model class we will use to train embeddings on song sequences.

In [10]:
# let's train this model using the playlists data
from gensim.models import Word2Vec

Train a Word2Vec model on the playlist sequences. Adjust vector size, window, and worker threads as needed.

In [11]:
# train the Word2Vec model
model = Word2Vec(
    playlists, vector_size=32, window=20, min_count=1, workers=4
)

Query the trained model for songs similar to a given song id. This returns the most similar song ids and scores.

In [12]:
song_id = 2172

# ask the model for the most similar songs to it
model.wv.most_similar(str(song_id), topn=10)        

[('2842', 0.9946203827857971),
 ('1785', 0.9908266663551331),
 ('10052', 0.990791916847229),
 ('6641', 0.9906285405158997),
 ('1922', 0.9904156923294067),
 ('1840', 0.9902911186218262),
 ('2156', 0.9890431761741638),
 ('2704', 0.988997757434845),
 ('5549', 0.9882001280784607),
 ('2117', 0.9875474572181702)]

Look up the metadata (title, artist) for the selected song id using the songs DataFrame.

In [13]:
# let's find out what this song id is
print(songs_df.iloc[song_id])

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object


Define a helper that fetches DataFrame rows for the model's top recommendations and print the recommendations for a song id.

In [16]:
# lets also find out the other similar songs
import numpy as np

def print_recommendations(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id), topn=10)
    )[:,0]
    return songs_df.iloc[similar_songs]

# extract recommendations
print_recommendations(2172)

,title,artist
id,,
2842,For Whom The Bell Tolls,Metallica
1785,Flying High Again,Ozzy Osbourne
10052,Between The Lines,Stone Temple Pilots
6641,Shout At The Devil,Motley Crue
1922,One,Metallica
1840,Mama I'm Coming Home,Ozzy Osbourne
2156,No More Tears,Ozzy Osbourne
2704,Over The Mountain,Ozzy Osbourne
5549,November Rain,Guns N' Roses
